# Retrieval Accuracy Demo: HyDE + Hybrid RRF + Reranking

Companion notebook to the LinkedIn article **"Your RAG Accuracy Problem
Isn't the Model. It's the Retrieval."**

This walks through the HR policy scenario from the article: an employee
asks "how many sick days do I get," and the correct, up-to-date answer
lives in a policy update document phrased nothing like the question. We
compare a naive single-method retrieval baseline against the full
three-fix pipeline (HyDE -> hybrid retrieval + RRF -> cross-encoder
reranking) on the same synthetic corpus.

All LLM and reranker calls are stubbed with lightweight synthetic
functions so this notebook runs end-to-end with no API key and no
downloaded model weights. Swap in `make_bge_reranker()` and a real
Claude API call for production use -- see `retrieval_accuracy.py`.


In [1]:
import sys
sys.path.insert(0, "..")

from retrieval_accuracy import (
    HybridRetriever,
    full_pipeline,
    hyde_expand,
    reciprocal_rank_fusion,
    rerank,
)
import numpy as np


## The synthetic corpus

A small HR document set: the current handbook, an outdated benefits
section, and the actual policy update that answers the question --
phrased in formal, legalistic language that shares almost no words with
how an employee would actually ask.


In [2]:
corpus = {
    "policy_update_q2": (
        "Revisions to Section 4.2 leave entitlements effective Q2. "
        "All full-time staff are entitled to twelve days of paid medical "
        "leave per calendar year, up from ten, replacing the prior "
        "entitlement schedule in its entirety."
    ),
    "handbook_section_4": (
        "Section 4: Leave Policy. Employees receive ten days of paid "
        "medical leave annually, accrued monthly, as outlined at time "
        "of hire."
    ),
    "handbook_section_7": (
        "Section 7: Workplace Conduct. All employees are expected to "
        "maintain professional conduct standards outlined in the code "
        "of ethics."
    ),
    "reimbursement_faq": (
        "How to submit a reimbursement claim for travel expenses "
        "incurred during approved business trips."
    ),
}

query = "how many sick days do I get"
print(f"Query: {query!r}")
print(f"Correct answer lives in: policy_update_q2")
print(f"Naive keyword overlap with query: "
      f"{set(query.lower().split()) & set(corpus['policy_update_q2'].lower().split())}")


Query: 'how many sick days do I get'
Correct answer lives in: policy_update_q2
Naive keyword overlap with query: {'days'}


Notice the query and the correct document share almost no words in
common -- "sick days" vs. "medical leave entitlements." This is exactly
the gap HyDE is designed to close, and exactly why naive keyword or
even plain semantic search can miss it.


## Stub embedding, LLM, and reranker functions

These stand in for a real embedding model, a real Claude API call, and
a real BGE cross-encoder. The embedding stub is deliberately weak on
paraphrase (bag-of-words hashing) so the baseline failure mode shows up
clearly, then HyDE + reranking recover it.


In [3]:
def bag_of_words_embed(text: str) -> np.ndarray:
    """
    A deliberately weak embedding: represents text as a fixed-size hashed
    bag-of-words vector. Captures exact word overlap well, captures
    paraphrase (synonyms like 'sick days' vs 'medical leave') poorly --
    which is realistic for cases where a general-purpose embedding model
    hasn't been tuned to a domain's specific vocabulary.
    """
    dim = 64
    vec = np.zeros(dim)
    for word in text.lower().split():
        idx = hash(word) % dim
        vec[idx] += 1.0
    return vec


doc_embeddings = {doc_id: bag_of_words_embed(text) for doc_id, text in corpus.items()}


def stub_llm_fn(prompt: str) -> str:
    """
    Stands in for a real Claude API call. Returns a hand-written
    hypothetical answer in the same style as the real policy document --
    this is what a real HyDE call to Claude would produce for this query.
    """
    return (
        "Revisions to leave entitlements state that employees are "
        "entitled to twelve days of paid medical leave per calendar "
        "year, effective the current quarter."
    )


def stub_rerank_score_fn(query: str, doc_text: str) -> float:
    """
    Stands in for a real BGE cross-encoder. Scores word overlap between
    the ORIGINAL query and each candidate, plus a small bonus for
    mentioning specific day counts -- a rough proxy for 'does this
    document actually answer the question,' which a real cross-encoder
    learns directly.
    """
    q_words = set(query.lower().split())
    d_words = set(doc_text.lower().split())
    overlap = len(q_words & d_words)
    specificity_bonus = 1.0 if any(ch.isdigit() for ch in doc_text) else 0.0
    return overlap + specificity_bonus


## Baseline: naive single-method dense retrieval

No HyDE, no hybrid fusion, no reranking -- just embed the raw query and
rank by cosine similarity, which is what most first-pass RAG
implementations do by default.


In [4]:
def cosine_sim(a, b):
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) or 1e-9
    return float(np.dot(a, b) / denom)


query_vec = bag_of_words_embed(query)
baseline_scores = {
    doc_id: cosine_sim(query_vec, vec) for doc_id, vec in doc_embeddings.items()
}
baseline_ranked = sorted(baseline_scores.items(), key=lambda x: x[1], reverse=True)

print("Baseline ranking (naive dense retrieval on raw query):")
for doc_id, score in baseline_ranked:
    flag = "  <-- correct answer" if doc_id == "policy_update_q2" else ""
    print(f"  {doc_id:22s} score={score:.3f}{flag}")


Baseline ranking (naive dense retrieval on raw query):
  reimbursement_faq      score=0.267
  policy_update_q2       score=0.199  <-- correct answer
  handbook_section_4     score=0.192
  handbook_section_7     score=0.140


With a weak embedding and no query expansion, the correct document
often doesn't come out on top -- the paraphrase gap between "sick days"
and "medical leave entitlements" costs it. If this were handed straight
to the LLM, it would likely answer using the outdated handbook section
instead, confidently and incorrectly.


## Fix 1 alone: HyDE query expansion


In [5]:
hypothetical = hyde_expand(query, stub_llm_fn)
print(f"Hypothetical answer used for retrieval:\n  {hypothetical!r}\n")

hyde_vec = bag_of_words_embed(hypothetical)
hyde_scores = {doc_id: cosine_sim(hyde_vec, vec) for doc_id, vec in doc_embeddings.items()}
hyde_ranked = sorted(hyde_scores.items(), key=lambda x: x[1], reverse=True)

print("Ranking after HyDE expansion:")
for doc_id, score in hyde_ranked:
    flag = "  <-- correct answer" if doc_id == "policy_update_q2" else ""
    print(f"  {doc_id:22s} score={score:.3f}{flag}")


Hypothetical answer used for retrieval:
  'Revisions to leave entitlements state that employees are entitled to twelve days of paid medical leave per calendar year, effective the current quarter.'

Ranking after HyDE expansion:
  policy_update_q2       score=0.778  <-- correct answer
  handbook_section_4     score=0.433
  reimbursement_faq      score=0.340
  handbook_section_7     score=0.327


Because the hypothetical answer is phrased in the same style as the
real policy document, it shares far more vocabulary with
`policy_update_q2` than the original question did -- the correct
document moves up.


## Full pipeline: HyDE + hybrid retrieval/RRF + reranking

Now the complete pipeline: HyDE-expanded query, fused dense + BM25
retrieval, then a final reranking pass against the *original* query
(reranking always uses the real question, since that's what actually
needs answering -- HyDE is a retrieval aid, not a replacement for the
user's intent).


In [6]:
final_results = full_pipeline(
    query=query,
    corpus=corpus,
    embed_fn=bag_of_words_embed,
    doc_embeddings=doc_embeddings,
    llm_fn=stub_llm_fn,
    rerank_score_fn=stub_rerank_score_fn,
    fused_top_k=4,
    final_top_n=3,
    use_hyde=True,
)

print("Final ranking after full pipeline:")
for r in final_results:
    flag = "  <-- correct answer" if r.doc_id == "policy_update_q2" else ""
    print(f"  {r.doc_id:22s} score={r.score:.2f}{flag}")
    print(f"    {r.text}")


Final ranking after full pipeline:
  policy_update_q2       score=2.00  <-- correct answer
    Revisions to Section 4.2 leave entitlements effective Q2. All full-time staff are entitled to twelve days of paid medical leave per calendar year, up from ten, replacing the prior entitlement schedule in its entirety.
  handbook_section_4     score=2.00
    Section 4: Leave Policy. Employees receive ten days of paid medical leave annually, accrued monthly, as outlined at time of hire.
  handbook_section_7     score=1.00
    Section 7: Workplace Conduct. All employees are expected to maintain professional conduct standards outlined in the code of ethics.


## Side-by-side comparison


In [7]:
def rank_of(doc_id, ranked_ids):
    for i, entry in enumerate(ranked_ids):
        current_id = entry[0] if isinstance(entry, tuple) else entry.doc_id
        if current_id == doc_id:
            return i + 1
    return None


target = "policy_update_q2"
print(f"Rank of correct document ({target}) at each stage:")
print(f"  Baseline (naive dense, raw query):        #{rank_of(target, baseline_ranked)}")
print(f"  + HyDE query expansion:                   #{rank_of(target, hyde_ranked)}")
print(f"  + Hybrid retrieval, RRF, and reranking:    #{rank_of(target, final_results)}")


Rank of correct document (policy_update_q2) at each stage:
  Baseline (naive dense, raw query):        #2
  + HyDE query expansion:                   #1
  + Hybrid retrieval, RRF, and reranking:    #1


## Takeaway

Each fix addresses a distinct failure mode:

- **HyDE** closes the vocabulary gap between how the question is asked
  and how the answer is written.
- **Hybrid retrieval + RRF** protects against either search method's
  individual blind spots (paraphrase vs. exact terms/numbers).
- **Reranking** does a final, more careful check against the real
  question, correcting any residual ordering issues from the first
  pass.

None of this touches the language model doing the final answer
generation. All of it changes what that model is shown before it
speaks -- which is where this class of RAG accuracy problem actually
lives.
